# Random Forests — Bagging, OOB Intuition, and Feature Importance

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026F_predictive_analytics_QM474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2027S_predictive_analytics_QM474/blob/main/notebooks/nb12_random_forests_importance_student.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain how **bootstrap aggregation (bagging)** plus **random feature subsets** turn high-variance single trees into stable ensembles, on both classification and regression cases.
2. Fit `RandomForestClassifier` and `RandomForestRegressor` and test, with `compare_paired` on identical folds, whether a forest earns a **worthwhile gain** over nb11's single tree and over the **nb09 reference** baselines.
3. Tune `n_estimators`, `max_features` and `max_depth` with a joint grid, and treat the grid's best cell as a **nominee** that must beat the default forest before it replaces it.
4. Read the **Out-of-Bag (OOB) score** as a free held-out estimate — and know when to trust it vs cross-validation.
5. Build the **four-method feature-importance reconciliation table** (linear-coefficient / impurity / permutation / drop-column), with permutation and drop-column measured on held-out folds — the course-wide reference table that nb13 and nb14 rely on for interpretation, and that you will reuse in your own final-project interpretation section.

---


> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises** — Exercise 1 on the classification track (RF tuning) and Exercise 2 on the regression track (RF tuning). Complete both before submitting your notebook.

---

## 💼 Why This Matters

The single decision tree from nb11 is high-variance: a small change in the training data flips the root split, and from there the entire tree restructures. Both the State Health Department's review board and HomeValue Analytics' CFO asked the same follow-up question after seeing the tree: *"What happens to this model if you retrain it on next month's data?"*

The honest answer is *"a different tree"*, which is not the answer either stakeholder wanted. **Random forests** fix this by training many trees on bootstrap samples with random feature subsets, then averaging their predictions. No single tree dominates; the forest votes. Variance drops, predictions stabilize, and you keep the ability to handle non-linearity that linear baselines cannot.

The cost is a loss of single-path interpretability — you can no longer trace a flowchart from root to leaf because there are dozens or hundreds of them. In exchange you get:

- **More stable predictions** under retraining (averaging cancels much of the tree-to-tree variance; how much depends on how correlated the trees are).
- **Built-in OOB estimates** — every tree was trained on a bootstrap sample, so the \~37% of training rows left out of each tree are a free held-out set for that tree.
- **Four different views of feature importance** that, when they disagree, tell you something interesting about the data.

Today's question for each stakeholder is the same one nb11 asked of the single tree: does the forest earn a **worthwhile gain** over the model they run today? Score challenger and incumbent on the identical folds with `compare_paired`, and read the corrected interval of the per-fold gain against the margin band both stakeholders declared in nb08 — **base 0.01** (ROC-AUC for the review board, R² for HomeValue's CFO), with **0.005** and **0.02** as the sensitivity check. Only an interval whose lower end clears the base margin promotes the forest; *unresolved* or *practically equivalent* leaves the simpler incumbent in place. No promise is made here about how either case comes out — that is what Section 9 computes. That discipline is the bar nb14's selection ceremony will enforce on the full candidate roster.

> **A question that often comes up here:** *"If a forest is just an average of trees, why do I need the four-method importance table?"* Because the average of trees is opaque. You cannot read 100 flowcharts. The importance table is the diagnostic that lets you answer the **"what is the model paying attention to?"** question without re-deriving it from the trees.

---


## 1. Setup — Imports, References, Helpers

The setup cell imports the random-forest estimators for both cases, locks `RANDOM_SEED = 474`, defines the **nb09 reference pipelines** (`reference_clf` = LogReg(C=1.0); `reference_reg` = OLS), and pastes in nb08's comparison tools unchanged: `compare_paired`, the margin bands `DELTAS_AUC` and `DELTAS_R2`, and the helpers that print and plot a comparison (`show_comparison`, `fold_table`, `plot_fold_scores`, `plot_gain_intervals`). Two helpers are new for this notebook: `plot_importance_bars` and `plot_importance_heatmap`.


> 💡 **Gemini Prompt:** "Set up imports for sklearn RandomForestClassifier, RandomForestRegressor, DecisionTreeClassifier, DecisionTreeRegressor, LogisticRegression, LinearRegression, permutation_importance, cross_val_score, RandomizedSearchCV, StratifiedKFold, KFold, clone, get_scorer, load_breast_cancer, fetch_california_housing, StandardScaler, Pipeline and scipy.stats. Set RANDOM_SEED = 474. Define reference_clf and reference_reg as the nb09 baseline pipelines. Paste the course's compare_paired(scores_a, scores_b, margin) function from nb08 exactly as written and define DELTAS_AUC and DELTAS_R2 as {'small': 0.005, 'base': 0.01, 'large': 0.02}. Define helpers: plot_importance_bars (horizontal bar chart with optional error bars) and plot_importance_heatmap (heatmap of feature ranks across methods, showing untested cells as a dash)."
>
> **After running, verify:**
> - [ ] `RANDOM_SEED = 474`, `reference_clf` and `reference_reg` defined
> - [ ] `compare_paired` and both margin bands printed back, with `base = 0.01`
> - [ ] All helpers callable; no import errors


In [ ]:
# Setup — imports, seed, nb09 references, plot helpers, the paired comparison and the margin band
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import (train_test_split, cross_val_score, StratifiedKFold, KFold,
                                     RandomizedSearchCV)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import get_scorer
from sklearn.inspection import permutation_importance
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)

# --- Course color convention ---
CLF_COLOR = '#1f77b4'
REG_COLOR = '#ff7f0e'
GREY      = '#999999'

# --- nb09 reference models: each stakeholder's current champion ---
reference_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(C=1.0, random_state=RANDOM_SEED, max_iter=5000))
])
reference_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('reg',    LinearRegression())
])

# --- compare_paired: the course's model-comparison rule from nb08 (pasted verbatim) ---
# A is the challenger, B the current champion. Both score arrays must come from ONE splitter object.
def compare_paired(scores_a, scores_b, margin):
    """Paired comparison on identical folds; higher score = better."""
    d = np.asarray(scores_a) - np.asarray(scores_b)
    k = len(d)
    t = stats.t.ppf(0.975, k - 1)
    se = d.std(ddof=1) * np.sqrt(1/k + 1/(k - 1))   # Nadeau–Bengio correction
    lo, hi = d.mean() - t*se, d.mean() + t*se
    if lo > margin:
        verdict = "A is better by a worthwhile amount"
    elif hi < -margin:
        verdict = "B is better by a worthwhile amount"
    elif -margin < lo and hi < margin:
        verdict = "Practically equivalent: choose the simpler model"
    else:
        verdict = "Unresolved: apply the default you declared"
    return d, d.mean(), (lo, hi), verdict

# --- The margin band each stakeholder declared in nb08, before any score was computed ---
# base decides the champion; small and large are the sensitivity check (same numbers through nb14).
DELTAS_AUC = {'small': 0.005, 'base': 0.01, 'large': 0.02}   # MedScreen review board — ROC-AUC
DELTAS_R2  = {'small': 0.005, 'base': 0.01, 'large': 0.02}   # HomeValue CFO — R²
DELTA_AUC  = DELTAS_AUC['base']
DELTA_R2   = DELTAS_R2['base']

def show_comparison(challenger, champion, scores_a, scores_b, deltas, metric):
    """Print one paired comparison: fold gains (description), corrected interval, verdict, band check."""
    d, gain, (lo, hi), verdict = compare_paired(scores_a, scores_b, deltas['base'])
    print(f"{challenger} (A, challenger) vs {champion} (B, current champion) — {metric}")
    print("  per-fold gains A − B (description only): " + ', '.join(f"{x:+.4f}" for x in d))
    print(f"  mean gain {gain:+.4f}; interval corrected for fold overlap [{lo:+.4f}, {hi:+.4f}]")
    print(f"  base margin δ = {deltas['base']}: {verdict}")
    band = {name: compare_paired(scores_a, scores_b, m)[3] for name, m in deltas.items()}
    same = len(set(v.startswith('A is better') for v in band.values())) == 1
    print("  band check: " + '; '.join(f"{n} {deltas[n]} → {v.split(':')[0]}" for n, v in band.items())
          + ("  → robust across the band" if same else "  → margin-sensitive"))
    return d, gain, (lo, hi), verdict

# --- Fold-score summary for ONE model: the five fold scores, their mean and their SD (no interval) ---
def fold_table(score_dict):
    rows = []
    for name, s in score_dict.items():
        s = np.asarray(s)
        rows.append({'model': name, **{f'fold {i+1}': v for i, v in enumerate(s)},
                     'mean': s.mean(), 'sd': s.std(ddof=1)})
    return pd.DataFrame(rows)

def plot_fold_scores(ax, score_dict, ylabel, title, colors=None):
    """Dot strip: each model's five fold scores and their mean — no error bars, no interval."""
    for j, (name, s) in enumerate(score_dict.items()):
        c = (colors or {}).get(name, GREY)
        ax.scatter(np.full(len(s), j) + np.linspace(-0.08, 0.08, len(s)), s, color=c, s=45,
                   edgecolor='black', zorder=3)
        ax.hlines(np.mean(s), j - 0.25, j + 0.25, color='black', linewidth=2, zorder=4)
    ax.set_xticks(range(len(score_dict)))
    ax.set_xticklabels(list(score_dict), fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

def plot_gain_intervals(ax, rows, margin, xlabel, title):
    """rows: list of (label, d, (lo, hi)). Dots = per-fold gains; bar = corrected interval; shaded = ±δ."""
    ax.axvspan(-margin, margin, color=GREY, alpha=0.15, label=f'±δ = ±{margin}')
    ax.axvline(0, color='black', linewidth=1)
    for j, (label, d, (lo, hi)) in enumerate(rows):
        ax.hlines(j, lo, hi, color='black', linewidth=3)
        ax.scatter(d, np.full(len(d), j), color=REG_COLOR if 'R²' in xlabel else CLF_COLOR,
                   s=35, edgecolor='black', zorder=3)
        ax.scatter([np.mean(d)], [j], marker='D', color='white', edgecolor='black', s=60, zorder=4)
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([r[0] for r in rows], fontsize=9)
    ax.set_xlabel(xlabel)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.3, axis='x')

# --- Helper (NEW): horizontal importance bar chart with optional error bars ---
def plot_importance_bars(importances, names, ax, errors=None, color=CLF_COLOR,
                         title='Feature importance', top_n=15):
    """Horizontal bar chart, sorted descending. errors optional (e.g., fold-to-fold SD)."""
    df = pd.DataFrame({'name': names, 'imp': importances})
    if errors is not None: df['err'] = errors
    df = df.sort_values('imp', ascending=True).tail(top_n)
    if errors is not None:
        ax.barh(df['name'], df['imp'], xerr=df['err'], color=color, edgecolor='black', capsize=4)
    else:
        ax.barh(df['name'], df['imp'], color=color, edgecolor='black')
    ax.axvline(0, color='black', linewidth=1)
    ax.set_xlabel('Importance')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

# --- Helper (NEW): rank-disagreement heatmap (4-method reconciliation); untested cells shown as "–" ---
def plot_importance_heatmap(rank_df, ax, title='Feature rank across methods', top_n=15):
    """rank_df: rows = features, columns = methods; values = rank (1 = most important), NaN = not tested."""
    best_rank = rank_df.min(axis=1)
    keep = best_rank.nsmallest(top_n).index
    sub = rank_df.loc[keep].sort_values(rank_df.columns[0])
    im = ax.imshow(np.ma.masked_invalid(sub.values.astype(float)), cmap='RdYlGn_r', aspect='auto',
                   vmin=1, vmax=rank_df.shape[0])
    ax.set_xticks(range(sub.shape[1])); ax.set_xticklabels(sub.columns, rotation=20, ha='right')
    ax.set_yticks(range(sub.shape[0])); ax.set_yticklabels(sub.index)
    for i in range(sub.shape[0]):
        for j in range(sub.shape[1]):
            v = sub.values[i, j]
            ax.text(j, i, '–' if pd.isna(v) else int(v), ha='center', va='center', fontsize=9)
    ax.set_title(title, fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax, label='Rank (1 = most important; – = not tested)', shrink=0.7)

print(f"✓ RANDOM_SEED = {RANDOM_SEED}")
print("✓ nb09 references: reference_clf, reference_reg")
print(f"✓ Margin band (base decides): ROC-AUC {DELTAS_AUC}, R² {DELTAS_R2}")
print("✓ Helpers: compare_paired, show_comparison, fold_table, plot_fold_scores, plot_gain_intervals, "
      "plot_importance_bars, plot_importance_heatmap")


**Reading the output:**

The comparison toolkit is the same one nb08 and nb11 used, pasted unchanged so every notebook reaches the same verdict from the same fold scores. `fold_table` and `plot_fold_scores` *describe* a model: its five fold scores, their mean and their SD — no error bars around a single model's score. `compare_paired` and `plot_gain_intervals` *decide*: they subtract two models' scores fold by fold, put an interval corrected for the overlap between folds around the mean gain, and read it against the margin. `show_comparison` prints both, plus whether the verdict survives the small and large margins. The two new helpers serve Sections 7 and 8: `plot_importance_bars` draws the stakeholder-facing permutation chart, and `plot_importance_heatmap` renders the four-method reconciliation table.

The two reference pipelines `reference_clf` and `reference_reg` are exactly what survived nb09's comparisons on the two datasets — each stakeholder's current champion, and the incumbent at the bottom of Section 9's ladder.

**Key takeaway:** All helpers and both nb09 references are defined once and reused across every section; their signatures match nb11's, so muscle memory transfers across notebooks.

---


## 2. Load Both Datasets — One 80/20 Split, Two Locked Envelopes

Same discipline as nb11: each case is split **once**, 80% training / 20% test, with `random_state=RANDOM_SEED`, and the test envelopes stay sealed until nb14's ceremony. Every model decision runs on the training rows through 5-fold CV — stratified for classification, plain `KFold` for regression — and there is no separate validation set. The variable suffixes `_clf` / `_reg` keep the two namespaces unconfusable.


In [ ]:
# --- Classification track: Wisconsin breast cancer ---
data_clf = load_breast_cancer(as_frame=True)
X_clf, y_clf = data_clf.data, data_clf.target

# One 80/20 split (the same first split as nb08, nb09 and nb11): X_train for all model work.
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_SEED, stratify=y_clf
)
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# --- Regression track: California Housing ---
data_reg = fetch_california_housing(as_frame=True)
X_reg, y_reg = data_reg.data, data_reg.target

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_SEED
)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print("=== CLASSIFICATION (Wisconsin Breast Cancer) ===")
print(f"  Train: {len(X_train_clf):>6} | Test: {len(X_test_clf):>5} (LOCKED until nb14)")
print()
print("=== REGRESSION (California Housing) ===")
print(f"  Train: {len(X_train_reg):>6} | Test: {len(X_test_reg):>5} (LOCKED until nb14)")


**Reading the output:**

455 training patients and 16,512 training tracts — the same 80/20 split as nb11, because the same `random_state=RANDOM_SEED` produces the same partition, and the same splitters produce the same five folds. That is the whole point of locking the seed: a model scored here can be compared fold by fold with nb11's tree, and when nb14's ceremony names a champion, the chain of CV evidence from nb11 → nb12 → nb13 → nb14 sits on identical rows.

**Key takeaway:** Same split and same folds as nb11 → results are directly comparable, fold by fold.

---


## 3. From Single Tree to Forest — The Bagging Idea

nb11 ended with an honest finding: a single decision tree is interpretable but **high-variance**. Train it on a slightly different set of patients or census tracts and the root split can flip to a different feature, the leaves rearrange, and the predictions move. That fragility is the problem random forests solve, and the fix has a beautifully simple shape: **average many noisy estimates, and much of the noise cancels out**.

Imagine a HomeValue analyst asking ten colleagues to estimate the median house value of one neighborhood. Each colleague works from a different set of comparable sales, with their own habits and blind spots. Any one estimate is noisy. But if the analyst takes the **average** across all ten, the colleague-specific noise tends to cancel, and the average usually sits closer to the true value than most of the individual estimates. That is the entire idea behind **bagging** — short for *bootstrap aggregating* — applied to decision trees. Each "analyst" in the forest is one tree, trained on a slightly different sample of the data, and the forest's prediction is the average (for regression) or majority vote (for classification) across all the trees.

The "slightly different sample" part is what bootstrap does. The figure below is the canonical bootstrap-resampling diagram from the ISLR textbook:

<center>
<img src="https://raw.githubusercontent.com/davi-moreira/2026F_predictive_analytics_QM474/main/notebooks/figures/5_11-1.png" width="640" alt="Bootstrap resampling — draw B samples with replacement from the original training data, fit one model per sample, aggregate the B estimates.">
</center>

The original training data on the left has three rows. To create one **bootstrap sample**, draw three rows *with replacement* — each draw is independent, and the same row can be picked more than once. In the top sample (Z*¹) row 3 was drawn twice and row 2 was never drawn. The middle sample (Z*²) drew rows 2, 3, and 1 — every original row appears exactly once. The bottom sample (Z*ᴮ) drew row 2 twice and row 1 once. Each sample has the same size as the original training set, but the *composition* is different. Each sample produces its own estimate `α̂*ᵇ`; the final ensemble combines all `B` of them.

Two numbers fall out of this procedure that are worth memorizing:

- **About 63% of the original rows appear** in any one bootstrap sample (some appear multiple times, others not at all).
- **The remaining \~37% are out-of-bag** for that sample — they form a free held-out set you can use to validate the tree without spending any of the locked test data. Section 6 will turn this into the OOB diagnostic.

In bagging we draw `B` bootstrap samples — typically 100 to 500 — fit one decision tree on each, and combine their predictions at the end. The intuition the HomeValue analyst used scales directly: averaging `B` noisy estimators reduces the variance of the ensemble's prediction far below the variance of any individual tree. Trees fit on different bootstraps are not perfectly independent (they share most of the original rows), so the variance does not fall as fast as the textbook `1/B` formula would suggest — but the direction is always the same. More bootstraps means less wobble.

The plot below makes the bootstrap part visible on real data. Five bootstrap samples are drawn from the breast-cancer **training set you just loaded in Section 2**, and the distribution of one feature (`mean radius`) is plotted under each. The five curves cluster around the original distribution but diverge in details — that variability across bootstraps is exactly what averaging a forest cancels out.

In [ ]:
# Bootstrap KDEs — visualize what each base learner actually sees on real data.
# Uses X_train_clf['mean radius'] from the 455-patient training pool loaded in Section 2.
X_show = X_train_clf['mean radius'].values
rng = np.random.default_rng(RANDOM_SEED)

fig, ax = plt.subplots(figsize=(11, 5))
xs = np.linspace(X_show.min(), X_show.max(), 400)
for b in range(5):
    boot = rng.choice(X_show, size=len(X_show), replace=True)
    bw = 1.06 * boot.std() * len(boot) ** (-1/5)
    kde = np.exp(-0.5 * ((xs[:, None] - boot[None, :]) / bw) ** 2).sum(axis=1) / (len(boot) * bw * np.sqrt(2*np.pi))
    ax.plot(xs, kde, alpha=0.7, linewidth=2, label=f'Bootstrap {b+1}')

bw0 = 1.06 * X_show.std() * len(X_show) ** (-1/5)
kde0 = np.exp(-0.5 * ((xs[:, None] - X_show[None, :]) / bw0) ** 2).sum(axis=1) / (len(X_show) * bw0 * np.sqrt(2*np.pi))
ax.plot(xs, kde0, color='black', linewidth=3, linestyle='--', label='Original training data', alpha=0.8)

ax.set_xlabel('mean radius')
ax.set_ylabel('density')
ax.set_title('Five bootstrap samples of the breast-cancer training set — every base learner sees a slightly different slice',
             fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✓ Each bootstrap is similar but not identical — a different tree fits a different shape.")
print("✓ Random forests average across dozens to hundreds of these slices; the forest's variance is far below any single tree's.")


**Reading the output:**

All five bootstrap curves cluster around the original training distribution (the dashed black line) but diverge in details — different peaks, different tails, different local densities. A decision tree fit on each bootstrap will pick slightly different split thresholds in `mean radius` and slightly different child structures, even though the underlying training set is the same 455-patient pool you loaded in Section 2.

The variance you see across these five curves is the variance you saw in nb11's `max_depth=20` overfit row — large, fold-dependent, fragile under retraining. The random forest's prediction is the average (for regression) or majority vote (for classification) over **dozens or hundreds** of trees like these. Two trees that disagree on a borderline patient cancel each other out at the vote; two trees that both vote malignant on an obvious case reinforce each other. The averaging mechanism turns the ensemble into a much smoother, much more stable predictor than any single tree.

> **A question that often comes up here:** *"if bootstrap is just sampling with replacement, why does that produce different trees?"* Two reasons. First, each tree's training set is genuinely different — fit one tree on the same data twice and you get the same tree, but fit it on two different bootstraps and you get two different trees, with different root splits, different thresholds, and different leaves. Second, the trees' errors are partially **uncorrelated** — when tree A is wrong on a particular patient, tree B is often right, and the average of their predictions is closer to truth than either individual prediction. That is the same variance-reduction-by-averaging logic the HomeValue analyst used at the top of this section, applied to a model ensemble instead of a panel of colleagues.

**Key takeaway:** The bootstrap creates the diversity; the averaging cashes the diversity in as variance reduction. Section 4 measures the cash-in directly — the single tree's fold-to-fold spread against a 50-tree forest's under identical conditions, and the paired gain between the two.

---

## 4. Single Tree vs Random Forest — Paired

The headline comparison: each case's **single tree from nb11** against a **50-tree random forest** fit at the *same depth*. Classification uses `max_depth=3` (the review board's one-page flowchart); regression uses `max_depth=5` (the legal team's five-question limit). Same depth, same data, same CV folds — the only thing that changes is *one tree → fifty trees averaged*, so this section isolates what bagging buys. Fifty trees keep the illustration quick; Section 5 tunes the forest properly. This is a test of the bagging idea, not the stakeholder's decision: whether a forest replaces the nb09 reference is Section 9's question.

Read the output in the usual two layers. The **description** — each model's five fold scores, mean and SD — shows whether averaging tightened the fold-to-fold spread. The **decision** — `compare_paired(forest, tree, δ)` — says whether the forest's gain over the tree is worthwhile, practically nothing, or unresolved.


> 💡 **Gemini Prompt:** "Score DecisionTreeClassifier(max_depth=3, random_state=474) and RandomForestClassifier(n_estimators=50, max_features='sqrt', max_depth=3, random_state=474, n_jobs=-1) on X_train_clf with cv_clf and scoring='roc_auc', and DecisionTreeRegressor(max_depth=5, random_state=474) and RandomForestRegressor(n_estimators=50, max_features=0.5, max_depth=5, random_state=474, n_jobs=-1) on X_train_reg with cv_reg and scoring='r2'. Use the same cv object for both models on a case. Print each model's five fold scores, mean and SD with fold_table — no interval around a single model. Then run show_comparison(forest = A, tree = B) on each case with DELTAS_AUC / DELTAS_R2, and plot the fold scores with plot_fold_scores and the paired gains with plot_gain_intervals."
>
> **After running, verify:**
> - [ ] Both models on a case use the same `cv_clf` / `cv_reg` object
> - [ ] Tables show five fold scores, mean and SD per model — no interval around a single model
> - [ ] Each case prints per-fold gains, a corrected interval, and one of the four verdicts at δ = 0.01
> - [ ] The forest's fold SD is smaller than the tree's on classification


In [ ]:
# Single tree vs forest at the SAME depth — the bagging effect on identical folds.
tree_clf   = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED)
forest_clf = RandomForestClassifier(n_estimators=50, max_features='sqrt', max_depth=3, random_state=RANDOM_SEED, n_jobs=-1)
tree_reg   = DecisionTreeRegressor(max_depth=5, random_state=RANDOM_SEED)
forest_reg = RandomForestRegressor(n_estimators=50, max_features=0.5, max_depth=5, random_state=RANDOM_SEED, n_jobs=-1)

# One splitter per case (cv_clf, cv_reg) for both models → fold i holds the same rows for both.
clf_scores = {
    'Single Tree (depth=3)':  cross_val_score(tree_clf,   X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'Random Forest (50×3)':   cross_val_score(forest_clf, X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
}
reg_scores = {
    'Single Tree (depth=5)':  cross_val_score(tree_reg,   X_train_reg, y_train_reg, cv=cv_reg, scoring='r2',      n_jobs=-1),
    'Random Forest (50×5)':   cross_val_score(forest_reg, X_train_reg, y_train_reg, cv=cv_reg, scoring='r2',      n_jobs=-1),
}

print("=== CLASSIFICATION — 5-fold CV ROC-AUC at depth 3: fold scores, mean, SD ===")
print(fold_table(clf_scores).to_string(index=False))
print()
print("=== REGRESSION — 5-fold CV R² at depth 5: fold scores, mean, SD ===")
print(fold_table(reg_scores).to_string(index=False))

print()
d_clf, gain_clf, ci_clf, verdict_clf = show_comparison(
    'Random Forest (50×3)', 'Single Tree (depth=3)',
    clf_scores['Random Forest (50×3)'], clf_scores['Single Tree (depth=3)'], DELTAS_AUC, 'ROC-AUC')
print()
d_reg, gain_reg, ci_reg, verdict_reg = show_comparison(
    'Random Forest (50×5)', 'Single Tree (depth=5)',
    reg_scores['Random Forest (50×5)'], reg_scores['Single Tree (depth=5)'], DELTAS_R2, 'R²')

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
plot_fold_scores(axes[0, 0], clf_scores, '5-fold CV ROC-AUC', 'Classification — fold scores at depth 3 (bar = mean)',
                 colors={'Random Forest (50×3)': CLF_COLOR})
plot_fold_scores(axes[0, 1], reg_scores, '5-fold CV R²', 'Regression — fold scores at depth 5 (bar = mean)',
                 colors={'Random Forest (50×5)': REG_COLOR})
plot_gain_intervals(axes[1, 0], [('forest − tree', d_clf, ci_clf)], DELTA_AUC,
                    'Gain in ROC-AUC (forest − tree)', 'Classification — paired gain vs ±δ')
plot_gain_intervals(axes[1, 1], [('forest − tree', d_reg, ci_reg)], DELTA_R2,
                    'Gain in R² (forest − tree)', 'Regression — paired gain vs ±δ')
fig.suptitle('Bagging at a fixed depth — the fold spread tightens on classification; the paired gain is the comparison',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


**Reading the output:**

The two cases tell different but complementary stories about what bagging buys you — and one of them is a lesson about what fold wins can and cannot prove.

**Classification case (Wisconsin Breast Cancer, depth 3).** The single tree's fold scores run from **0.859 to 0.993** (mean **0.940**, SD **0.051**). The 50-tree forest at the same depth scores between **0.956 and 0.999** (mean **0.986**, SD **0.018**) — about a third of the tree's spread. That tightening is the variance-reduction-by-averaging mechanism from Section 3, visible on real folds: with only 455 training patients, the single tree's splits depend on which rows land in each fold, and averaging 50 trees fit on 50 bootstraps cancels much of that wander. Now the paired decision. The forest beats the tree on **every one of the five folds**, by +0.122, +0.034, +0.003, +0.038 and +0.033 — a mean gain of **+0.046** ROC-AUC. And yet the corrected interval is **[−0.038, +0.129]**: ***unresolved***, at every margin in the band. One fold (+0.122) carries more than half of the total gain and the other four are modest, so the gains are too uneven for five folds to rule out no gain at all. Fold wins are description; the interval decides. Here the honest sentence is *"the forest looks better on every fold, but these five folds cannot show a worthwhile gain."*

**Regression case (California Housing, depth 5).** The single tree averages a CV R² of **0.611** (SD **0.015**) and the 50-tree forest at the same depth **0.678** (SD **0.011**). Here the spread barely changes: on 16,512 training tracts a depth-5 tree is already stable, so bagging has little variance left to remove. The gain is all in the mean, and it is remarkably even across folds: +0.059 to +0.074 R², mean **+0.067**, corrected interval **[+0.057, +0.078]** — ***A is better by a worthwhile amount***, robust across the band. Even gains make a tight interval; that is why five folds are enough here and not on the classification case.

**The combined picture.** Averaging trees helps on both cases, but in different ways. On the small classification set it mainly tightens the fold-to-fold spread, and the gain over the tree is too uneven to certify. On the large regression set the tree was already stable, and the forest adds a consistent, worthwhile gain of about 0.07 R².

> **A question that often comes up here:** *"If 50 trees already average to a better mean than one tree, why not 500 or 1000?"* Two reasons. First, the marginal gain from each additional tree shrinks fast — Sections 5 and 6 show the score flattening as `n_estimators` grows. Second, averaging has a floor: bootstrap samples share most of their rows, so the trees are correlated, and the variance of their average stops falling once that shared part is all that is left. Past that point more trees buy you fit time, not predictive lift. `max_features` is the lever that controls how correlated the trees are — and therefore how low that floor sits — which is why Section 5 tunes it together with the tree count and depth.

**Key takeaway:** At a fixed depth, the forest tightens the classification fold spread to about a third of the tree's and adds a worthwhile, even gain of about 0.07 R² on regression. On classification it wins every fold and the comparison is still unresolved — the clearest demonstration yet that the corrected interval, not the count of winning folds, decides.

---


## 5. Tuning Random Forests — Joint `n_estimators × max_features × max_depth` Grid

Random forests have three knobs worth tuning together: **`n_estimators`** (how many trees in the ensemble), **`max_features`** (how many features each split considers), and **`max_depth`** (how deep each tree is allowed to grow). All three shape the bias/variance trade-off — **and they interact**, so tuning them in isolation can mislead you. The right tool is a **joint 3D grid**, the same idea you will see in nb13 §6 for gradient boosting.

- More `n_estimators` → lower variance, eventual plateau, no overfitting risk (more trees just smooth the average).
- Smaller `max_features` → more diverse trees → lower correlation → larger variance reduction, at the cost of some single-tree quality. scikit-learn's defaults are `'sqrt'` for classification and `1.0` (all features) for regression.
- Larger `max_depth` → more single-tree capacity → can capture non-linear structure that shallow trees cannot. Deeper trees overfit individually, but bagging averages much of that variance away.

The grid pairs **`n_estimators ∈ {50, 100, 200}`** × **`max_features ∈ {'sqrt', 0.5, None}`** × **`max_depth ∈ {3, 5, None}`** — nb11's carried depths (3 and 5) and scikit-learn's default of growing until pure (`None`) — and runs each of the 27 combinations through the same 5-fold CV on both cases. Every cell keeps its five fold scores. The figure renders one 3×3 heatmap per `(case, max_depth)` pair, with each cell showing the **CV mean** and the **fold SD** (the spread, not an interval).

How a grid's result becomes a decision is the important part. The cell with the largest mean is only a **nominee**. Its score is optimistic: it was picked out of 27 candidates on the same five folds that scored it, so part of its lead is luck that would not repeat on new rows. So the nominee faces the **family's default** — scikit-learn's own `RandomForestClassifier()` / `RandomForestRegressor()` settings, the forest you would get without any tuning — with `compare_paired` on the same folds, and replaces it only on a worthwhile gain. No tiebreakers, no reading the means for "close enough": the paired interval decides, and the forest that comes out of this section is the one the rest of the notebook (and nb13 and nb14) carries forward.


> 💡 **Gemini Prompt:** "Run a 3D grid sweep — n_estimators in [50, 100, 200] × max_features in ['sqrt', 0.5, None] × max_depth in [3, 5, None] — for both cases using 5-fold CV, keeping each cell's five fold scores in a dictionary. Classification: RandomForestClassifier(random_state=474, n_jobs=-1) with cv_clf and scoring='roc_auc'. Regression: RandomForestRegressor(random_state=474, n_jobs=-1) with cv_reg and scoring='r2'. Render a 3×2 figure: rows are max_depth values (3, 5, None); columns are the cases (classification in Blues, regression in Oranges). Each panel is a 3×3 heatmap of (max_features × n_estimators) annotated with the mean and the fold SD. Mark the cell with the largest mean per case (the nominee) with a red rectangle. Then score the default forest (RandomForestClassifier(random_state=474) / RandomForestRegressor(random_state=474)) on the same folds and run show_comparison(nominee = A, default = B). Carry the nominee forward only if the verdict is 'A is better by a worthwhile amount'; otherwise carry the default."
>
> **After running, verify:**
> - [ ] Six panels arranged 3 rows × 2 columns (max_depth × case), each 3 × 3
> - [ ] Each cell shows the CV mean and the fold SD — no ± interval
> - [ ] One red rectangle per case, on the cell with the largest mean, and the nominee labelled optimistic
> - [ ] `compare_paired` output for nominee vs default on each case, and the forest carried forward printed


In [ ]:
# Joint 3D sweep n_estimators × max_features × max_depth — 5-fold CV on both cases.
# Every cell keeps its five fold scores, so any cell can face any other model on identical folds.
from matplotlib.patches import Rectangle

n_grid_rf  = [50, 100, 200]
mf_grid_rf = ['sqrt', 0.5, None]
md_grid_rf = [3, 5, None]
mf_labels_rf = ["'sqrt'", '0.5', 'None']
md_labels_rf = ['3', '5', 'None']

grid_folds_clf, grid_folds_reg = {}, {}
for md in md_grid_rf:
    for mf in mf_grid_rf:
        for n in n_grid_rf:
            grid_folds_clf[(md, mf, n)] = cross_val_score(
                RandomForestClassifier(n_estimators=n, max_features=mf, max_depth=md, random_state=RANDOM_SEED, n_jobs=-1),
                X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1)
            grid_folds_reg[(md, mf, n)] = cross_val_score(
                RandomForestRegressor(n_estimators=n, max_features=mf, max_depth=md, random_state=RANDOM_SEED, n_jobs=-1),
                X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1)

def _arrays(folds):
    mean = np.zeros((3, 3, 3)); sd = np.zeros((3, 3, 3))
    for k, md in enumerate(md_grid_rf):
        for i, mf in enumerate(mf_grid_rf):
            for j, n in enumerate(n_grid_rf):
                s = folds[(md, mf, n)]; mean[k, i, j] = s.mean(); sd[k, i, j] = s.std(ddof=1)
    return mean, sd

clf_mean, clf_sd = _arrays(grid_folds_clf)
reg_mean, reg_sd = _arrays(grid_folds_reg)

# The NOMINEE per case is simply the cell with the largest CV mean (no tiebreaker).
nominee_clf = max(grid_folds_clf, key=lambda key: grid_folds_clf[key].mean())
nominee_reg = max(grid_folds_reg, key=lambda key: grid_folds_reg[key].mean())
idx = lambda key: (md_grid_rf.index(key[0]), mf_grid_rf.index(key[1]), n_grid_rf.index(key[2]))

clf_vmin, clf_vmax = float(clf_mean.min()), float(clf_mean.max())
reg_vmin, reg_vmax = float(reg_mean.min()), float(reg_mean.max())

fig, axes = plt.subplots(3, 2, figsize=(14, 13))
for k, md in enumerate(md_grid_rf):
    for case_idx, (mean_slc, sd_slc, vmin, vmax, nom, title, cmap) in enumerate([
        (clf_mean[k], clf_sd[k], clf_vmin, clf_vmax, idx(nominee_clf), f"Classification — max_depth={md_labels_rf[k]}", 'Blues'),
        (reg_mean[k], reg_sd[k], reg_vmin, reg_vmax, idx(nominee_reg), f"Regression — max_depth={md_labels_rf[k]}",     'Oranges'),
    ]):
        ax = axes[k, case_idx]
        ax.imshow(mean_slc, cmap=cmap, aspect='auto', vmin=vmin, vmax=vmax)
        for i in range(mean_slc.shape[0]):
            for j in range(mean_slc.shape[1]):
                ax.text(j, i, f'{mean_slc[i, j]:.4f}\nSD {sd_slc[i, j]:.4f}',
                        ha='center', va='center', fontsize=9,
                        color='black' if mean_slc[i, j] < vmin + 0.8 * (vmax - vmin) else 'white',
                        fontweight='bold')
        if nom[0] == k:
            ax.add_patch(Rectangle((nom[2] - 0.5, nom[1] - 0.5), 1, 1, fill=False, edgecolor='red', linewidth=3))
        ax.set_xticks(range(len(n_grid_rf))); ax.set_xticklabels([str(n) for n in n_grid_rf])
        ax.set_yticks(range(len(mf_grid_rf))); ax.set_yticklabels([f'mf={lbl}' for lbl in mf_labels_rf])
        ax.set_xlabel('n_estimators'); ax.set_title(title, fontsize=11, fontweight='bold')

fig.suptitle('Joint n_estimators × max_features × max_depth grid — CV mean and fold SD; red box = the nominee (largest mean) per case',
             fontsize=12, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

# Range of CV means inside each max_depth panel (the heatmaps above, in numbers)
print("CV-mean range per max_depth panel:")
for k, md in enumerate(md_grid_rf):
    print(f"  max_depth={md_labels_rf[k]:>4}:  classification {clf_mean[k].min():.4f}–{clf_mean[k].max():.4f}   "
          f"regression {reg_mean[k].min():.4f}–{reg_mean[k].max():.4f}")
print()

# --- D3: the nominee faces the family default (sklearn's own RandomForest settings) on the same folds ---
default_forest_clf = RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1)   # 100 trees, 'sqrt', no depth limit
default_forest_reg = RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1)    # 100 trees, all features, no depth limit
default_folds_clf = cross_val_score(default_forest_clf, X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1)
default_folds_reg = cross_val_score(default_forest_reg, X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1)

def _cfg(key):
    md, mf, n = key
    return f"n_estimators={n}, max_features={mf!r}, max_depth={md}"

print(f"Nominee — classification: {_cfg(nominee_clf)}  → mean {grid_folds_clf[nominee_clf].mean():.4f} (optimistic: picked from 27 cells on these folds)")
print(f"Nominee — regression:     {_cfg(nominee_reg)}  → mean {grid_folds_reg[nominee_reg].mean():.4f} (optimistic: picked from 27 cells on these folds)")
print()
_, _, _, v_nom_clf = show_comparison('Grid nominee', 'Default forest', grid_folds_clf[nominee_clf], default_folds_clf, DELTAS_AUC, 'ROC-AUC')
print()
_, _, _, v_nom_reg = show_comparison('Grid nominee', 'Default forest', grid_folds_reg[nominee_reg], default_folds_reg, DELTAS_R2, 'R²')

# The forest this notebook carries forward: the nominee only on a worthwhile gain; otherwise the default.
def _forest_from(key, Est):
    md, mf, n = key
    return Est(n_estimators=n, max_features=mf, max_depth=md, random_state=RANDOM_SEED, n_jobs=-1)

if v_nom_clf.startswith('A is better'):
    forest_clf_final, forest_clf_label = _forest_from(nominee_clf, RandomForestClassifier), f'Random Forest ({_cfg(nominee_clf)})'
else:
    forest_clf_final, forest_clf_label = default_forest_clf, 'Random Forest (default)'
if v_nom_reg.startswith('A is better'):
    forest_reg_final, forest_reg_label = _forest_from(nominee_reg, RandomForestRegressor), f'Random Forest ({_cfg(nominee_reg)})'
else:
    forest_reg_final, forest_reg_label = default_forest_reg, 'Random Forest (default)'
print()
print(f"✓ Forest carried forward — classification: {forest_clf_label}")
print(f"✓ Forest carried forward — regression:     {forest_reg_label}")


**Reading the output:**

The figure shows **six panels arranged 3 rows × 2 columns**: rows are `max_depth ∈ {3, 5, None}`, columns are the two business cases. Each panel is a 3×3 heatmap of `(max_features × n_estimators)` with the CV mean and the fold SD in every cell, and the red rectangle marks each case's **nominee** — the cell with the largest mean. The printed lines below the figure give each panel's range and the comparison that actually decides.

**Classification case — State Health Department (Wisconsin Breast Cancer).** All 27 cells fall between **0.979 and 0.990** ROC-AUC — the depth-3 panel 0.982–0.990, depth 5 0.980–0.988, unlimited depth 0.979–0.989. The nominee is `(n_estimators=200, max_features='sqrt', max_depth=3)` at **0.9897**. Against scikit-learn's default forest (`n_estimators=100, max_features='sqrt'`, no depth limit; 0.9883), its per-fold gains are +0.0015, −0.0010, −0.0013, 0.0000 and +0.0077 — a mean of **+0.0014** with a corrected interval of **[−0.0055, +0.0083]**, entirely inside ±0.01: ***practically equivalent***, so the simpler model — the untuned default — stays. No margin in the band would promote the nominee, so the result is robust.

**Regression case — HomeValue Analytics (California Housing).** Depth dominates the picture. The three depth panels form three clearly separated bands: **0.458–0.559** at depth 3, **0.625–0.678** at depth 5, and **0.800–0.818** with unlimited depth — every unlimited-depth cell is at least 0.12 R² above every depth-5 cell. The forest needs deep trees on this data; nb11's readable depth caps were a requirement for a *single* tree, and a forest is not read as a flowchart anyway. The nominee is `(n_estimators=200, max_features='sqrt', max_depth=None)` at **0.8178**. Against the default forest (`n_estimators=100`, all 8 features per split, no depth limit; 0.8036) the per-fold gains are all positive, **+0.011 to +0.018**, mean **+0.0142**, corrected interval **[+0.0079, +0.0205]**. The interval clears zero but its lower end sits below the base margin of 0.01: ***unresolved***, so the declared default applies and the default forest stays. At the small margin (0.005) the nominee would win, so this verdict is **margin-sensitive** — worth reporting to the CFO, who decided in advance that 0.01 R² is the gain worth a switch.

**What the rest of the course carries forward:** the **default forest** on both cases — `RandomForestClassifier(random_state=474)` and `RandomForestRegressor(random_state=474)`. Sections 6–9, nb13 and nb14 all use it.

> **A question that often comes up here:** *"The regression nominee beat the default on all five folds. Isn't keeping the default throwing away a better model?"* It may be a better model — the evidence cannot tell yet. Two things hold you back. First, the nominee's score is optimistic: it is the largest of 27 means computed on the same five folds, and the largest of many noisy numbers is biased upward (the winner's curse of model selection; Cawley and Talbot, 2010). Second, even taken at face value, its gain over the default has an interval whose lower end is 0.0079 — below the 0.01 the CFO said would justify a switch. If you want an honest estimate of what *"tune the forest with this grid"* is worth, the going-further method is **nested cross-validation**: run the whole 27-cell grid again inside each of the five outer folds, on that fold's training rows only, and score the configuration it picks on the outer fold's held-out rows. Those five outer scores measure the tuning *procedure* without the selection bias — at the price of re-running the grid five times, which is why the course uses the single-loop nominee with an explicit "optimistic" label instead.

**Key takeaway.** Tune `n_estimators × max_features × max_depth` together, keep every cell's fold scores, and treat the best cell as a nominee that must beat the default forest on the same folds. Here neither nominee earned a worthwhile gain — practically equivalent on MedScreen, unresolved (and margin-sensitive) on HomeValue — so the **default forest** is the forest the course carries forward.

---


## 6. Out-of-Bag (OOB) Score — A Free Held-Out Estimate

Section 3 noted that every bootstrap sample leaves about 37% of the training rows un-sampled. Those un-sampled rows are **out-of-bag** for the tree that was fit on that bootstrap — that specific tree has never seen them. That gives the random forest a useful party trick: for any training row, predict it using **only** the trees that did not see it in their bootstrap. The result is a held-out score on the entire training set, computed during fitting, at zero extra cost. That is the **OOB score**.

The OOB number usually agrees with the 5-fold CV mean to within a small fraction of the CV fold spread. When the two diverge meaningfully, something is often wrong with the assumption that rows are independent: typically the data has hidden structure (groups, time, panel effects) that bootstrap sampling does not respect. Neither of today's datasets has that structure, so OOB and CV should track each other closely.

The plot below traces both signals as `n_estimators` grows, for the forest Section 5 carried forward. The OOB curve is a single value per setting (no folds to average); the CV curve is the 5-fold mean with the fold SD as error bars.


> 💡 **Gemini Prompt:** "For n_estimators in [25, 50, 100, 200, 300], take the forest carried forward from Section 5 (forest_clf_final / forest_reg_final), set oob_score=True and the tree count with clone(...).set_params(...), fit it on X_train_clf / X_train_reg and record oob_score_. At each setting also run 5-fold cross_val_score with cv_clf (scoring='accuracy') or cv_reg (scoring='r2') and record the CV mean and SD. Print a table and render a 1×2 figure (OOB curve and CV mean with SD error bars vs n_estimators), one panel per case."
>
> **After running, verify:**
> - [ ] OOB and CV track each other within a small gap across the sweep on both cases
> - [ ] On classification, the OOB value sits inside the CV fold SD at every setting
> - [ ] The table prints OOB, CV mean and CV SD for both cases


In [ ]:
# OOB vs CV for the forest carried forward from Section 5 — same 5-fold cv_clf / cv_reg as before.
n_est_grid_oob = [25, 50, 100, 200, 300]

clf_oob, clf_cv_mean, clf_cv_sd = [], [], []
for n in n_est_grid_oob:
    m = clone(forest_clf_final).set_params(n_estimators=n, oob_score=True)
    m.fit(X_train_clf, y_train_clf)
    clf_oob.append(m.oob_score_)                                   # OOB accuracy
    s = cross_val_score(m, X_train_clf, y_train_clf, cv=cv_clf, scoring='accuracy', n_jobs=-1)
    clf_cv_mean.append(s.mean()); clf_cv_sd.append(s.std(ddof=1))

reg_oob, reg_cv_mean, reg_cv_sd = [], [], []
for n in n_est_grid_oob:
    m = clone(forest_reg_final).set_params(n_estimators=n, oob_score=True)
    m.fit(X_train_reg, y_train_reg)
    reg_oob.append(m.oob_score_)                                   # OOB R²
    s = cross_val_score(m, X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1)
    reg_cv_mean.append(s.mean()); reg_cv_sd.append(s.std(ddof=1))

print(pd.DataFrame({'n_estimators': n_est_grid_oob,
                    'clf OOB acc': clf_oob, 'clf CV acc': clf_cv_mean, 'clf CV SD': clf_cv_sd,
                    'reg OOB R²': reg_oob, 'reg CV R²': reg_cv_mean, 'reg CV SD': reg_cv_sd}).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
ax = axes[0]
ax.plot(n_est_grid_oob, clf_oob, marker='o', linewidth=2, color=CLF_COLOR, label='OOB accuracy')
ax.errorbar(n_est_grid_oob, clf_cv_mean, yerr=clf_cv_sd, marker='s', linewidth=2,
            capsize=6, color=GREY, label='5-fold CV (mean ± fold SD)')
ax.set_xlabel('n_estimators'); ax.set_ylabel('Accuracy')
ax.set_title('Classification — OOB vs CV', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(n_est_grid_oob, reg_oob, marker='o', linewidth=2, color=REG_COLOR, label='OOB R²')
ax.errorbar(n_est_grid_oob, reg_cv_mean, yerr=reg_cv_sd, marker='s', linewidth=2,
            capsize=6, color=GREY, label='5-fold CV (mean ± fold SD)')
ax.set_xlabel('n_estimators'); ax.set_ylabel('R²')
ax.set_title('Regression — OOB vs CV', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

fig.suptitle('OOB tracks CV closely — a free held-out estimate at zero extra fits',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n✓ OOB and CV curves should overlap. If they diverge, suspect data dependence (groups, time).")


**Reading the output:**

The two cases tell the same broad story — OOB and 5-fold CV track each other — but at different granularity, because the datasets have very different sizes. Both panels use the forest Section 5 carried forward.

**Classification case (Wisconsin Breast Cancer, 455 training patients).** Both signals sit between **0.952 and 0.963** accuracy across the sweep. At 25, 50 and 200 trees the two numbers are identical, and the largest gap is **0.004** (0.954 OOB vs 0.958 CV at 100 trees) — about a sixth of the CV fold SD, which runs from 0.024 to 0.028. Two different held-out schemes tell the same story here, which is a reassuring sanity check rather than a proof that either number is the forest's true accuracy. For the State Health Department, the practical implication is that the OOB number is a good fast working estimate for this forest.

**Regression case (California Housing, 16,512 training tracts).** Both signals rise from about **0.79–0.80 at 25 trees** to about **0.81 at 300 trees**. At 25 trees OOB trails CV by 0.010 (0.786 vs 0.796) — each row's OOB prediction then averages only the few trees that left it out. From 50 trees on, OOB sits *slightly above* CV, by 0.003 to 0.006 R² (0.811 vs 0.805 at 300 trees). That small, consistent lead has a mechanical reason: OOB predictions come from trees fit on bootstraps of all 16,512 rows, while each CV model is fit on only four-fifths of them. The CV fold SD here is about 0.01, so the two signals agree to within about half of the fold spread.

**The combined picture.** On both cases OOB delivers the agreement the bootstrap math promises when rows are independent. A defensible recipe: fit one forest with `oob_score=True`, read `oob_score_`, and use it as the fast working estimate while you explore. Spend the 5-fold CV compute when you need to compare a forest against another model — the per-fold gains that decide such a comparison exist only when both models ran the same folds, and one OOB number cannot give you them.

> **A question that often comes up here:** *"Would OOB still match CV if the dataset had time-series structure instead of independent rows?"* No. Bootstrap sampling draws rows freely; if observations have temporal order (today's row depends on yesterday's), the OOB rows for a tree may be temporally *adjacent* to its in-bag rows — leaking information across the OOB partition. CV with grouped or time-aware folds would catch this, but OOB silently agrees with itself. The diagnostic value of CV is precisely that it can fail visibly; OOB cannot. Use OOB on independent rows; reach for grouped or time-aware CV the moment the rows have any structure.

**Key takeaway.** OOB is the random forest's free held-out estimate. Trust it as a working number when OOB and CV agree within the fold spread — as they do on both cases here. Spend the CV compute when you need per-fold scores for a paired comparison, or when you suspect the rows are not independent.

---


## 7. Feature Importance — The Four-Method Reconciliation Table

The stakeholder question is always the same — *"which features matter most?"* — and the honest answer is that there are at least four reasonable ways to measure that, and they often disagree. The disagreement is not a flaw; it is the pedagogical payload. When all four methods agree on a feature, you can quote any of them to the stakeholder. When they disagree, you have to think about **what each method is actually measuring** before you decide what to report.

Each of the four methods is a different *question* you can ask about a fitted model. Walk through them one at a time before reading any rank table:

| # | Method | Question it answers | Measured on | Cost |
|---|---|---|---|---|
| 1 | Linear coefficient magnitude | What does the linear baseline rank as important? | the reference fit on `X_train` | Free (one model fit) |
| 2 | MDI (`feature_importances_`) | What does the forest rank as it builds its trees? | the forest fit on `X_train` | Free (side-effect of forest fit) |
| 3 | Permutation importance | What is lost when this column's values are scrambled? | each CV fold's held-out rows | Moderate (`n_repeats × n_features` re-scorings per fold) |
| 4 | Drop-column importance | What is lost when this column does not exist at all? | each CV fold's held-out rows | Expensive (one refit per feature per fold) |

**1. Linear coefficient magnitude** asks: *"in the linear model's view, how much does the prediction move when this feature changes by one standard deviation, holding everything else constant?"* You compute it by fitting the nb09 reference pipeline (the same `LogReg(C=1.0)` for classification or OLS for regression) on standardized features, and then reading off the **absolute value of each coefficient**. Standardizing matters here — without it, a feature measured in dollars and a feature measured in percentages would have artificially different coefficient sizes for reasons that have nothing to do with how important they actually are. Cost: one model fit, essentially free.

**2. MDI — Mean Decrease in Impurity** asks: *"every time this feature gets picked to split a node in any tree of the forest, how much does it shrink the impurity (gini for classification, MSE for regression) at that split?"* The forest sums those impurity drops across every split in every tree, weighted by how many training rows reached each split. It is the scikit-learn default — calling `forest.feature_importances_` after fitting returns the MDI vector. Cost: essentially free, because MDI is computed as a side-effect of fitting the forest. **One caveat:** MDI is measured on the training rows and tends to inflate features that have many distinct values (a continuous feature offers many split thresholds; a binary feature offers only one), so it slightly favours high-cardinality features even when they are not more informative.

**3. Permutation importance** asks: *"what happens if I take this one feature's column, randomly shuffle its values — keeping the column there, but breaking its connection to the target — and re-score the already-fitted model?"* If the feature was carrying signal, the scrambled version scores worse, and the size of the drop measures how much the feature mattered. Here it runs **inside the 5-fold loop**: in each fold the forest is fit on the fold's training rows and the shuffling is done on that fold's held-out rows (5 shuffles per feature), so the importance describes performance on rows the model never saw. The five fold-level results are averaged, and their fold-to-fold SD becomes the error bar on the §8 chart.

**4. Drop-column importance** asks: *"if this feature simply did not exist, and I **rebuilt** the forest without it, how much worse would the held-out score be?"* In each of the same five folds, the forest is refit without the column and scored on the fold's held-out rows; the importance is the average loss (full minus reduced). This is the most expensive method — one refit per feature per fold — but it is also the most stakeholder-aligned: the other three ask *"how important was this feature in the model we have?"*, while drop-column asks *"how much would we lose if we could not measure this feature at all?"* Because of the expense, we test only the top features by MDI (top-10 on classification, top-5 on regression); untested features show as "–" in the table rather than as a worst rank. And because the full and reduced forests were scored on the same folds, the drop-column result is itself a paired comparison — the right way to decide whether a measurement can be dropped.

The output below is a **rank heatmap**: rows are features, columns are methods, cell colour encodes the rank a method assigned to a feature (1 = most important). Rows that look uniformly dark across all four columns are robustly important. Rows that look mottled — different ranks across columns — are where the methods disagree, and the *pattern* of disagreement is itself the diagnostic.


> 💡 **Gemini Prompt:** "Write a function that computes four importance methods for one case: (1) absolute standardized coefficients of the nb09 reference fitted on X_train; (2) MDI from the forest carried forward from Section 5, fitted on X_train; (3) permutation importance inside the 5-fold loop — for each fold of cv, fit a clone of the forest on the fold's training rows and run permutation_importance on its held-out rows with n_repeats=5, then average across folds and keep the fold-to-fold SD; (4) drop-column importance in the same loop — refit the forest without each column on the fold's training rows and record full score minus reduced score on the held-out rows, for the top features by MDI only. Convert each method to ranks (1 = most important), leaving untested drop-column features as NaN. Run it on breast cancer (ROC-AUC, cv_clf, top 10 for drop-column) and California Housing (R², cv_reg, top 5), and render both rank tables with the plot_importance_heatmap helper."
>
> **After running, verify:**
> - [ ] Both cases produce a rank DataFrame with 4 columns (one per method)
> - [ ] Permutation and drop-column are computed on held-out fold rows, never on the rows the forest was fitted on
> - [ ] Untested drop-column cells show "–", not a rank
> - [ ] At least one feature has very different ranks across methods (the disagreement signal)


In [ ]:
# Helper: compute four-method importance ranks for a single case.
# Methods 3 and 4 run within the 5-fold loop: each fold fits on its training rows and measures
# on its held-out rows, so no importance is measured on rows the forest was fitted on.
def four_method_ranks(reference_pipeline, ref_step_name, forest_estimator,
                      X_train, y_train, scoring, cv, label,
                      drop_col_subset=None, n_repeats=5):
    """Returns (rank_df, perm_table, drop_folds, full_scores): ranks, permutation mean/SD, drop-column fold losses, full-model fold scores."""
    feat_names = list(X_train.columns)
    scorer = get_scorer(scoring)

    # Method 1: linear coefficient magnitude (standardized nb09 reference fitted on X_train)
    ref = clone(reference_pipeline).fit(X_train, y_train)
    coef = ref.named_steps[ref_step_name].coef_
    coef_mag = np.abs(coef.ravel() if coef.ndim > 1 else coef)

    # Method 2: MDI (impurity-based) from the forest fitted on all of X_train
    mdi = clone(forest_estimator).fit(X_train, y_train).feature_importances_

    # Methods 3 + 4 inside the 5-fold loop
    cols_to_drop = drop_col_subset if drop_col_subset is not None else feat_names
    perm_by_fold = []                              # one row of importances per fold
    full_scores, reduced_scores = [], {c: [] for c in cols_to_drop}
    for tr_idx, ho_idx in cv.split(X_train, y_train):
        X_tr, X_ho = X_train.iloc[tr_idx], X_train.iloc[ho_idx]
        y_tr, y_ho = y_train.iloc[tr_idx], y_train.iloc[ho_idx]
        model = clone(forest_estimator).fit(X_tr, y_tr)
        full_scores.append(scorer(model, X_ho, y_ho))
        perm = permutation_importance(model, X_ho, y_ho, scoring=scoring, n_repeats=n_repeats,
                                      random_state=RANDOM_SEED, n_jobs=-1)
        perm_by_fold.append(perm.importances_mean)
        for col in cols_to_drop:                    # drop-column: refit without the column, same fold
            reduced = clone(forest_estimator).fit(X_tr.drop(columns=[col]), y_tr)
            reduced_scores[col].append(scorer(reduced, X_ho.drop(columns=[col]), y_ho))
    perm_by_fold = np.array(perm_by_fold)
    perm_table = pd.DataFrame({'mean': perm_by_fold.mean(axis=0), 'fold_sd': perm_by_fold.std(axis=0, ddof=1)},
                              index=feat_names)
    full_scores = np.array(full_scores)
    drop_folds = {c: full_scores - np.array(reduced_scores[c]) for c in cols_to_drop}   # per-fold loss when dropped
    drop_imp = np.array([drop_folds[c].mean() if c in drop_folds else np.nan for c in feat_names])

    # Convert each method's importance to RANK (1 = most important); untested drop-column features stay NaN.
    def to_rank(imp):
        return pd.Series(imp, index=feat_names).rank(ascending=False, method='min', na_option='keep')

    rank_df = pd.DataFrame({
        'Linear coef': to_rank(coef_mag),
        'MDI':         to_rank(mdi),
        'Permutation': to_rank(perm_table['mean'].values),
        'Drop-column': to_rank(drop_imp),
    }, index=feat_names)
    print(f"=== {label} — top-10 features by linear coef rank (– = not tested by drop-column) ===")
    print(rank_df.sort_values('Linear coef').head(10).to_string(na_rep='–', float_format=lambda v: f'{v:.0f}'))
    return rank_df, perm_table, drop_folds, full_scores

# --- Classification: drop-column on the 10 features with the highest MDI (runtime) ---
mdi_quick = clone(forest_clf_final).fit(X_train_clf, y_train_clf).feature_importances_
top_clf_features = list(X_train_clf.columns[np.argsort(mdi_quick)[::-1][:10]])

ranks_clf, perm_clf, drop_folds_clf, full_folds_clf = four_method_ranks(
    reference_pipeline=reference_clf, ref_step_name='clf', forest_estimator=forest_clf_final,
    X_train=X_train_clf, y_train=y_train_clf, scoring='roc_auc', cv=cv_clf, label='CLASSIFICATION',
    drop_col_subset=top_clf_features)

# --- Regression: drop-column on the 5 features with the highest MDI (runtime) ---
mdi_quick_r = clone(forest_reg_final).fit(X_train_reg, y_train_reg).feature_importances_
top_reg_features = list(X_train_reg.columns[np.argsort(mdi_quick_r)[::-1][:5]])

ranks_reg, perm_reg, drop_folds_reg, full_folds_reg = four_method_ranks(
    reference_pipeline=reference_reg, ref_step_name='reg', forest_estimator=forest_reg_final,
    X_train=X_train_reg, y_train=y_train_reg, scoring='r2', cv=cv_reg, label='REGRESSION',
    drop_col_subset=top_reg_features)

print()
print("Drop-column loss per feature (full − reduced, mean over the 5 held-out folds):")
for lbl, folds in (('clf', drop_folds_clf), ('reg', drop_folds_reg)):
    print(f"  {lbl}: " + ', '.join(f"{c} {v.mean():+.4f}" for c, v in sorted(folds.items(), key=lambda kv: -kv[1].mean())))

# A removal decision is a model comparison: the forest WITHOUT the column (challenger) against the forest
# WITH it (current champion), on the same five folds — here for each case's top permutation feature.
for lbl, perm_tab, folds, full, deltas, metric in (
        ('MedScreen', perm_clf, drop_folds_clf, full_folds_clf, DELTAS_AUC, 'ROC-AUC'),
        ('HomeValue', perm_reg, drop_folds_reg, full_folds_reg, DELTAS_R2, 'R²')):
    top = perm_tab['mean'].idxmax()
    if top in folds:
        print()
        show_comparison(f'{lbl} forest without {top}', 'forest with every feature', full - folds[top], full, deltas, metric)


In [ ]:
# Render the four-method rank heatmaps ("–" = feature not in the drop-column subset)
fig, axes = plt.subplots(1, 2, figsize=(20, 9))
plot_importance_heatmap(ranks_clf, axes[0],
                        title='Classification (Wisconsin Breast Cancer) — feature rank across 4 methods',
                        top_n=15)
plot_importance_heatmap(ranks_reg, axes[1],
                        title='Regression (California Housing) — feature rank across 4 methods',
                        top_n=8)
plt.tight_layout()
plt.show()

print("\n💡 Rows that are uniformly green (low ranks across all 4 methods) are features every method agrees on.")
print("💡 Rows that are mottled (varying ranks) are where methods disagree — investigate why.")


**Reading the output:**

The pedagogical payload of this section is the disagreement *between* methods — not the cases where they agree. Reading each business case separately makes the pattern visible. Both tables describe the default forest Section 5 carried forward; permutation and drop-column are measured on held-out folds.

**Classification case (Wisconsin Breast Cancer).** **MDI** ranks `worst concave points`, `mean concave points`, `worst area` and `worst radius` at the top. **Permutation** on held-out rows broadly agrees on the family — `worst area` first, then `worst radius`, with `worst texture`, `worst concave points` and `area error` close behind. **Drop-column** tells a different story in size rather than order: among the ten features tested, the largest loss is **+0.0008** ROC-AUC (`mean concave points`), the next two are +0.0005 and +0.0001, and the other seven are zero or slightly negative — removing any one of them costs the forest essentially nothing. That is the **substitution effect**. The breast-cancer features contain several near-redundant variants of the same underlying signal — `worst radius`, `worst perimeter`, `worst area`, `mean radius`, `mean perimeter` all measure roughly *"how big is the tumour?"*. MDI and permutation credit whichever columns the fitted trees happened to use; drop-column asks *"if this column were gone, how much would the held-out score suffer?"*, and on a substitutable feature the answer is *"almost nothing, because the other size measurements step in."* The matched comparison printed above makes it a decision: the forest without `worst area` (the top permutation feature) against the forest with it gives a mean gain of −0.0001 and a corrected interval of [−0.0033, +0.0031] — *practically equivalent*. Neither method is wrong; they measure different things.

The fourth method, **linear-coefficient magnitude** from the standardized LogReg reference, lands somewhere else entirely: `radius error` is its #1 and `worst texture` its #2, while MDI ranks `radius error` 16th and permutation 12th. The logistic regression spreads credit across the correlated size measurements, so features that are *informative given the rest* — texture and the error-type measurements — rise to the top of the coefficient ranking.

**Regression case (California Housing).** The methods agree on the *set* of top features — `MedInc`, `Latitude`, `Longitude`, `AveOccup`, `HouseAge` — but disagree dramatically on the *order*. **MDI and permutation both rank `MedInc` (median income) first**, while **drop-column ranks it last of the five tested**: dropping `Longitude` or `Latitude` costs about **0.07 R²** each, dropping `AveOccup` about 0.009, and dropping `MedInc` *raised* the held-out R² by 0.013 on average. Before reading that as "stop collecting income data", look at the matched comparison: the forest without `MedInc` against the forest with it has per-fold gains from −0.006 to +0.025 and a corrected interval of **[−0.012, +0.038]** — ***unresolved***. The evidence shows neither that income can be dropped nor that it matters once location is in the model; the declared default keeps the full forest. What it does show is substitution again: location plus occupancy encode most of the same wealth signal, so the rebuilt forest compensates. `MedInc` is the strongest *individually* used feature (MDI, permutation) and also a *substitutable* one (drop-column).

The **linear-coef** ranking on regression puts `Latitude` and `Longitude` at the top — the standardized coefficients reward features whose marginal effect *given the others* is large, and location is not something income can stand in for in a linear model.

**The combined picture.** When all four methods rank a feature highly, you can quote it with confidence. When the methods disagree, the *pattern* of disagreement is the diagnostic:

- **MDI / Permutation high, Drop-column low** → the feature is informative but substitutable (you have correlated alternatives).
- **Linear-coef high, MDI / Permutation low** → the feature is informative *given the standardized rest* — often a signal the dominant cluster of features does not carry.
- **All four agree** → report it; the stakeholder can quote any method.

> **A question that often comes up here:** *"Which single method should I report to a stakeholder?"* Permutation importance measured on held-out rows, with its fold-to-fold spread as error bars, plus the four-method heatmap behind it for context. Permutation is model-agnostic (works on any fitted estimator, not just trees) and measures what the fitted model loses on unseen rows. Drop-column is the method to use for a *removal* decision, because it compares the model with and without the column on the same folds — and then the decision is a `compare_paired` verdict like any other; MDI is fast but measured on training rows and biased toward high-cardinality features; linear-coef is the simplest reference but does not capture non-linear interactions. None of the four is a causal effect.

**Key takeaway.** Methods agree when one feature dominates uniquely; methods disagree when features are correlated. On Wisconsin breast cancer the size-and-shape measurements substitute for one another, so dropping any one of them costs almost nothing, and linear-coef rewards the complementary texture and error features. On California Housing, MDI and permutation crown `MedInc` while drop-column ranks it last among the tested features, and the matched comparison for removing it is unresolved. **The disagreement is information about the data, not noise** — read the pattern before quoting any single ranking.

---


## 8. Permutation Importance — The Headline Plot for Stakeholders

Section 7 produced four different importance rankings and explained how to read the disagreement between them. Section 8 zooms into just one of those four — **permutation importance** — and visualizes it on its own with error bars. The reason for the spotlight is practical: of the four methods, permutation importance is the one you will most often put in front of a non-technical audience.

What does permutation importance actually do? You start with a fitted model. For each feature in turn, you take that feature's column and **randomly shuffle its values across the rows you are scoring**. The shuffling preserves the feature's distribution — same range, same shape, same fraction of high values — but breaks the link between that feature and the target. Then you re-score the model on the shuffled data. If the feature was carrying real signal, the shuffled score drops noticeably. If the feature was noise, the shuffled score barely changes. The **drop in score** is the permutation importance of that feature. Here it is measured on each CV fold's held-out rows, so it describes how the forest performs on rows it did not see.

Three reasons permutation importance is the right headline plot:

1. **Model-agnostic.** The same procedure works on any fitted estimator — today's random forest, the gradient-boosting machine you will meet in nb13, or a calibrated stacked model later in the course. Learn it once and reuse it everywhere.
2. **It answers the stakeholder question directly.** Stakeholders almost never ask *"how often does this feature appear in a tree's split?"* They ask *"how much would our predictions get worse if we could not measure this feature reliably?"* Permutation importance is close to that question.
3. **It comes with a visible spread.** Each fold gives its own estimate, so the five folds produce a mean and a fold-to-fold SD — the error bar on the plot. That bar describes how much the importance moved from fold to fold; it is not a confidence interval and it does not cover every source of uncertainty (for example, a different forest or a different choice of features).

The bar plot below reads `perm_clf` and `perm_reg`, which Section 7's helper already computed — no extra refits are needed. The two business cases are read separately because they produce very different importance profiles, which is itself part of the lesson.


> 💡 **Gemini Prompt:** "Print and plot the held-out permutation importances that Section 7 stored in perm_clf and perm_reg (columns 'mean' and 'fold_sd'). Render two side-by-side horizontal bar charts with error bars equal to the fold SD: left panel classification (top 15 features, CLF_COLOR), right panel regression (all 8 features, REG_COLOR), each sorted by mean, using the plot_importance_bars helper. Add a figure-level title."
>
> **After running, verify:**
> - [ ] Each bar's error bar is the fold-to-fold SD across the 5 held-out folds
> - [ ] Bars are sorted by mean importance on both panels
> - [ ] Classification panel shows 15 of the 30 features; regression panel shows all 8
> - [ ] A vertical line at zero makes negative means visible


In [ ]:
# Permutation importance on held-out folds — mean over the 5 folds, error bar = fold-to-fold SD
print("Classification — top 8 by held-out permutation importance:")
print(perm_clf.sort_values('mean', ascending=False).head(8).to_string())
print()
print("Regression — all 8 features:")
print(perm_reg.sort_values('mean', ascending=False).to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 9))
plot_importance_bars(perm_clf['mean'].values, perm_clf.index, axes[0],
                     errors=perm_clf['fold_sd'].values, color=CLF_COLOR,
                     title='Classification — held-out permutation importance ± fold SD', top_n=15)
plot_importance_bars(perm_reg['mean'].values, perm_reg.index, axes[1],
                     errors=perm_reg['fold_sd'].values, color=REG_COLOR,
                     title='Regression — held-out permutation importance ± fold SD', top_n=8)
fig.suptitle('Permutation importance on the 5 held-out folds — model-agnostic, with fold-to-fold spread per feature',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

The two business cases land on very different permutation-importance pictures. Both panels describe the default forest carried forward from Section 5, measured on the five held-out folds.

**Classification case (Wisconsin Breast Cancer, 30 features).** The bars on this panel are *small in absolute terms*: the largest is `worst area` at **0.0067** ROC-AUC, followed by `worst radius` (0.0054), `worst perimeter` (0.0049), `worst texture` (0.0041), `worst concave points` (0.0041), `area error` (0.0040) and `mean concave points` (0.0036). The forest is near the ceiling (CV ROC-AUC about 0.99), so there is little left to lose by scrambling any single column. And every one of those bars has a fold-to-fold SD about as large as its mean (`worst area` 0.0067 with SD 0.0059; `worst perimeter` 0.0049 with SD 0.0051), so the ranking among them would reshuffle on different folds.

The right message to the State Health Department's review board is twofold. First, **the size-and-shape "worst" measurements carry the screening signal as a group** — they fill the top of the chart. Second, **no single feature is irreplaceable**: the 30 cell measurements are highly correlated, so scrambling one column lets the trees lean on its neighbours, and Section 7's drop-column result says the same thing from the other direction. The model is robust to measurement error in any single column — good news for deployment. For a fuller substitution story, use the §7 four-method heatmap.

**Regression case (California Housing, 8 features).** The bars on this panel are far longer and far better separated. **`MedInc`** (median income) dominates: shuffling it on held-out rows drops R² by about **0.79** (fold SD 0.029) — most of the forest's R² of about 0.80. Below that, a clear second tier: **`Latitude`** (about **0.42**, SD 0.037), **`Longitude`** (**0.32**, SD 0.038) and **`AveOccup`** (**0.20**, SD 0.010). Then **`HouseAge`** (0.069), **`AveRooms`** (0.028), and two features near zero, **`AveBedrms`** (0.009) and **`Population`** (0.008). The top four sit many fold SDs above zero, so their order is stable from fold to fold. **HomeValue Analytics can put this chart in a stakeholder memo**: income first, then location, then occupancy density, then house age and dwelling size. One sentence must go with it, from §7: the fitted forest *relies* on income, but a forest rebuilt without income compensates through location — so "most important to this model" is not the same as "cannot be replaced".

**The combined picture.** Permutation importance is easy to read when a few features carry most of the signal — the regression case here. It *compresses* when many features are correlated and the model is near the ceiling — the classification case, where the bars are small and their spread is as large as the bars. The interpretive recipe: report the bars with their fold-to-fold spread, say plainly when the spread is as large as the differences, and fall back on the §7 heatmap to show the substitution pattern.

> **A question that often comes up here:** *"Why does permutation importance sometimes report negative values?"* Because shuffling a column that carries little signal can, by chance, leave the held-out score slightly *higher* — pure variation from the random shuffle and the particular fold. A negative mean says *"on these folds, this feature carries no detectable signal for this model"*. It does not say the feature is *harmful*, and it does not prove the feature is useless either — a correlated neighbour may simply be carrying the same information.

**Key takeaway.** Permutation importance on held-out rows is the right default plot when a few features clearly dominate (regression: `MedInc`, then location and occupancy). On near-ceiling, highly correlated problems (classification: the size-and-shape measurements) the bars compress and their fold-to-fold spread is as large as the bars themselves — report that honestly and supplement with the §7 heatmap. The permutation chart is the stakeholder default; the four-method heatmap is the diagnostic when permutation alone is not enough.

---


## 9. Comprehensive Model Comparison — The Champion Ladder

This closing section is the dress rehearsal for nb14's selection ceremony. Three candidates per business case, scored on **identical** cross-validation folds, climbed as a **ladder**: start from the model the stakeholder runs today, visit the challengers in order of complexity, and let a challenger replace the current champion **only on a worthwhile gain**.

- **The incumbent: the nb09 linear reference** — `LogReg(C=1.0)` in a `StandardScaler` pipeline for classification, scaled OLS for regression. nb11 tested the single tree against it on both cases and both comparisons came out *unresolved*, so the reference is still each stakeholder's champion as this section starts.
- **Challenger 1: nb11's single decision tree** — depth 3 on classification, depth 5 on regression (the readable trees). Included so the ladder shows its full log, even though nb11 already ran this step.
- **Challenger 2: the forest Section 5 carried forward** — the default forest on both cases, because neither grid nominee earned a worthwhile gain over it.

Each step is one `compare_paired(challenger, current champion, δ)` on the same folds, read against the base margin δ = 0.01 (ROC-AUC for the review board, R² for HomeValue's CFO). A worthwhile gain promotes the challenger; *unresolved* keeps the current champion (the default the stakeholders declared); *practically equivalent* keeps the simpler model, which on this ladder is always the current champion. The band check reports whether the final champion would change at the small or large margin. The Reading-the-output below names the two champions explicitly so you can defend them in writing to the State Health Department and HomeValue Analytics teams.


> 💡 **Gemini Prompt:** "Score three candidates per case on the same folds: for classification reference_clf, DecisionTreeClassifier(max_depth=3, random_state=474) and the forest carried forward from Section 5, with cv_clf and scoring='roc_auc'; for regression reference_reg, DecisionTreeRegressor(max_depth=5, random_state=474) and the carried forest, with cv_reg and scoring='r2'. Print each case's fold_table. Then climb a ladder per case: champion = the nb09 reference; for each challenger in order (tree, then forest) run show_comparison(challenger = A, champion = B) with DELTAS_AUC / DELTAS_R2 and promote the challenger only if the verdict starts with 'A is better'. Print the champion after each step and at the end, and plot the fold scores (plot_fold_scores) and every ladder step's corrected interval (plot_gain_intervals)."
>
> **After running, verify:**
> - [ ] Every candidate on a case uses the same `cv_clf` / `cv_reg` object
> - [ ] Each ladder step prints per-fold gains, the corrected interval, the verdict and the champion after the step
> - [ ] A challenger is promoted only on "A is better by a worthwhile amount"
> - [ ] The final champion for each case is printed


In [ ]:
# Champion ladder per case — the nb09 reference is the incumbent; challengers in order of complexity.
# Every candidate runs on the same splitter (cv_clf / cv_reg), so every comparison is paired.
single_tree_clf = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED)
single_tree_reg = DecisionTreeRegressor(max_depth=5, random_state=RANDOM_SEED)

clf_compare = {
    'nb09 reference: LogReg(C=1.0)': cross_val_score(reference_clf,    X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'Single Tree (depth=3, nb11)':   cross_val_score(single_tree_clf,  X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    forest_clf_label:                cross_val_score(forest_clf_final, X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
}
reg_compare = {
    'nb09 reference: OLS':           cross_val_score(reference_reg,    X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
    'Single Tree (depth=5, nb11)':   cross_val_score(single_tree_reg,  X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
    forest_reg_label:                cross_val_score(forest_reg_final, X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
}

print("=== CLASSIFICATION — 5-fold CV ROC-AUC: fold scores, mean, SD ===")
print(fold_table(clf_compare).to_string(index=False))
print()
print("=== REGRESSION — 5-fold CV R²: fold scores, mean, SD ===")
print(fold_table(reg_compare).to_string(index=False))

def run_ladder(scores, ladder, deltas, metric):
    """Incumbent first; a challenger replaces the current champion only on a worthwhile gain."""
    champion, rows = ladder[0], []
    for challenger in ladder[1:]:
        print()
        d, gain, ci, verdict = show_comparison(challenger, champion, scores[challenger], scores[champion], deltas, metric)
        promoted = verdict.startswith('A is better')
        rows.append((f'{challenger} − {champion}', d, ci))
        if promoted:
            champion = challenger
        print(f"  → champion now: {champion}")
    return champion, rows

print("\n######## MedScreen ladder (ROC-AUC, base δ = 0.01) ########")
champion_clf, rows_clf = run_ladder(clf_compare, list(clf_compare), DELTAS_AUC, 'ROC-AUC')
print("\n######## HomeValue ladder (R², base δ = 0.01) ########")
champion_reg, rows_reg = run_ladder(reg_compare, list(reg_compare), DELTAS_R2, 'R²')
print()
print(f"MedScreen champion: {champion_clf}")
print(f"HomeValue champion: {champion_reg}")

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
plot_fold_scores(axes[0, 0], clf_compare, '5-fold CV ROC-AUC', 'MedScreen — fold scores (bar = mean)')
plot_fold_scores(axes[0, 1], reg_compare, '5-fold CV R²', 'HomeValue — fold scores (bar = mean)')
plot_gain_intervals(axes[1, 0], rows_clf, DELTA_AUC, 'Gain in ROC-AUC (challenger − champion)', 'MedScreen ladder — each step vs ±δ')
plot_gain_intervals(axes[1, 1], rows_reg, DELTA_R2, 'Gain in R² (challenger − champion)', 'HomeValue ladder — each step vs ±δ')
fig.suptitle('Dress rehearsal for nb14 — the incumbent stays unless a challenger earns a worthwhile gain',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


**Reading the output:**

Two business cases, two ladders, two champions — each decided step by step by the corrected interval of a paired gain.

**Classification case (Wisconsin Breast Cancer — State Health Department).** The nb09 LogReg(C=1.0) averages a CV ROC-AUC of **0.994** (SD 0.007), nb11's single tree **0.940** (SD 0.051), and the default forest **0.988** (SD 0.014). *Step 1, tree vs LogReg:* mean gain −0.054, corrected interval [−0.143, +0.035] — ***unresolved***, exactly as in nb11, so LogReg stays. *Step 2, forest vs LogReg:* per-fold gains −0.012, +0.0005, −0.002, −0.0005 and −0.015, a mean gain of **−0.006** with a corrected interval of **[−0.019, +0.007]**. The interval reaches past −0.01 on one side and stays below +0.01 on the other: ***unresolved*** at the base margin (it would read *practically equivalent* only at the large margin, 0.02). Unresolved means the declared default: the incumbent stays. The champion is the same at all three margins. **Verdict: the State Health Department keeps the nb09 LogReg(C=1.0).** Notice also why no challenger could do much better on this evidence: two of LogReg's folds already score 0.9995, so on those folds no model can gain more than about 0.0005. The oncologists asked for a model they can explain to a patient; the standardized LogReg coefficients, read alongside the four-method importance table from §7, give them that.

**Regression case (California Housing — HomeValue Analytics).** OLS averages a CV R² of **0.604** (SD 0.019), the depth-5 tree **0.611**, and the default forest **0.804** (SD 0.012). *Step 1, tree vs OLS:* mean gain +0.006, interval [−0.011, +0.024] — ***unresolved***, so OLS stays (the same result as nb11 §7; the tree never displaced OLS). *Step 2, forest vs OLS:* per-fold gains between **+0.183 and +0.211** R², mean **+0.199**, corrected interval **[+0.180, +0.218]** — ***A is better by a worthwhile amount***, and robust across the band: the lower end clears even the large margin nine times over. **Verdict: HomeValue Analytics promotes the random forest** (`RandomForestRegressor(random_state=474)`, the default forest Section 5 carried forward) over OLS. In words the CFO can use: *"Against the model we run today, on the same five folds, the forest raises R² by about 0.20, with an interval corrected for fold overlap of [+0.18, +0.22] — far above the 0.01 we set before comparing."*

> **A question that often comes up here:** *"If the forest does not beat LogReg on classification, why was it worth running today?"* Three reasons. First, you produced the four-method importance reconciliation that the LogReg coefficients alone cannot give you on highly correlated features — a real interpretability deliverable, independent of which model ends up shipping. Second, the forest's regression-case promotion is decisive — one of your two business cases now has a defensible non-linear champion. Third, and most importantly, **a challenger that cannot show a worthwhile gain leaves the simpler model in place** — that is a *result*, not a failure. Knowing that the breast-cancer data leaves almost no ranking headroom above a linear model is itself useful information for the screening-tool stakeholders, and nb13 (gradient boosting) and nb14 (the selection ceremony) will keep testing this verdict with more candidates.

**Key takeaway.** Two cases, two ladders, both defensible. **HomeValue: the forest replaces OLS** — a worthwhile gain of about 0.20 R², robust across the band; the chain is simply **OLS → forest**, because nb11's tree never displaced OLS. **MedScreen: LogReg(C=1.0) stays** — the forest's comparison with it is unresolved, and so is the tree's. nb14 will run the same ladder with the full candidate roster (adding gradient boosting from nb13 and a regularized linear variant), and nb13 begins by putting gradient boosting against exactly these two champions.

---


## 📝 PAUSE-AND-DO Exercise 1 (clf, 5 minutes) — Tune the Classification Forest

**Task:** Find the best `(n_estimators, max_features)` combination for `RandomForestClassifier` on Wisconsin breast cancer, and test whether it earns a place over the default forest.

**Instructions:**
1. Sweep a 3×3 grid: `n_estimators ∈ [50, 100, 200]` × `max_features ∈ ['sqrt', 0.3, None]` (other settings at their defaults), keeping each cell's five fold scores from `cv_clf`.
2. For each combination, compute the 5-fold CV ROC-AUC mean and SD.
3. Render the means as a heatmap with values annotated and the best cell starred.
4. Treat the best cell as a **nominee** and compare it with the default forest (`max_features='sqrt'`, `n_estimators=100` — one of your nine cells) using `compare_paired(nominee_scores, default_scores, DELTA_AUC)`.
5. Write 3 short findings: which dial moved the score most, did the nominee earn its place, and why is the nominee's score optimistic?

---


> 💡 **Gemini Prompt:** "Grid-search RandomForestClassifier(random_state=474, n_jobs=-1) over n_estimators=[50,100,200] × max_features=['sqrt',0.3,None] using cross_val_score with cv_clf and scoring='roc_auc' on X_train_clf, keeping each cell's five fold scores. Print tables of the mean and SD, draw a 3×3 heatmap of means with the best cell starred, then run show_comparison with the best cell as A and the ('sqrt', 100) cell — scikit-learn's default forest — as B, using DELTAS_AUC."
>
> **After running, verify:**
> - [ ] Heatmap has 9 cells with mean values annotated and one star
> - [ ] The nominee is printed and labelled optimistic
> - [ ] `compare_paired` output shows per-fold gains, a corrected interval and one of the four verdicts


In [ ]:
# YOUR SOLUTION CODE HERE
# Tune RandomForestClassifier over (n_estimators, max_features) using 5-fold CV ROC-AUC (cv_clf).
# The best cell is a nominee: compare it with the default forest using compare_paired.


## 📝 PAUSE-AND-DO Exercise 2 (reg, 5 minutes) — Tune the Regression Forest

**Task:** Search the same `(n_estimators, max_features)` space for `RandomForestRegressor` on California Housing, and test whether the best configuration earns a place over the default forest.

**Instructions:**
1. Use the space `n_estimators ∈ [50, 100, 200]` × `max_features ∈ ['sqrt', 0.3, None]` (other settings at their defaults).
2. Forests of fully grown trees on 16,512 rows are slow on free Colab, so **shrink the search, never the folds**: run `RandomizedSearchCV(..., n_iter=4, cv=cv_reg, scoring='r2', random_state=474)`, which scores 4 of the 9 cells on the same five folds.
3. Print each sampled cell's mean R², fold SD and fit time.
4. Treat the best sampled cell as a **nominee** and compare its five fold scores with the default forest's (`RandomForestRegressor(random_state=474)`, scored on `cv_reg`) using `compare_paired(..., DELTA_R2)`; report the nominee's CV RMSE in USD.
5. Write 3 short findings: did the regression case prefer a different `max_features` than classification, and what should HomeValue do with the verdict?

---


> 💡 **Gemini Prompt:** "Run RandomizedSearchCV on RandomForestRegressor(random_state=474, n_jobs=-1) over n_estimators=[50,100,200] and max_features=['sqrt',0.3,None] with n_iter=4, cv=cv_reg, scoring='r2' and random_state=474, on X_train_reg. From cv_results_ print each sampled cell's mean test score, the SD of its five split scores and its mean fit time. Take the best sampled cell's five split scores as the nominee, score RandomForestRegressor(random_state=474) on cv_reg, and run show_comparison(nominee = A, default = B) with DELTAS_R2. Print the nominee's CV RMSE in USD (multiply by 100,000)."
>
> **After running, verify:**
> - [ ] Exactly 4 sampled cells printed, each scored with 5 folds
> - [ ] The nominee is printed and labelled optimistic
> - [ ] `compare_paired` output shows per-fold gains, a corrected interval and one of the four verdicts
> - [ ] The nominee's CV RMSE printed in USD


In [ ]:
# YOUR SOLUTION CODE HERE
# Search RandomForestRegressor over (n_estimators, max_features) with RandomizedSearchCV(n_iter=4, cv=cv_reg).
# The best sampled cell is a nominee: compare it with the default forest using compare_paired.


## 10. Wrap-Up — Key Takeaways

**What landed today:**

1. **Bagging + random feature subsets reduce variance — where there is variance to reduce.** At a fixed depth, the 50-tree forest cut the classification fold SD to about a third of the tree's (0.018 vs 0.051 on 455 training patients), while on regression (16,512 tracts) the depth-5 tree was already stable and the forest's gain was all in the mean: about +0.07 R², worthwhile and even across folds. On classification the forest won all five folds against the tree and the comparison was still *unresolved* — fold wins describe, the corrected interval decides.

2. **A grid's best cell is a nominee, not a champion.** The 27-cell grid's best cell faced scikit-learn's default forest on the same folds. On MedScreen it was *practically equivalent*; on HomeValue *unresolved* (and margin-sensitive). Neither earned a worthwhile gain, so the course carries **the default forest** on both cases — and the nominee's score was optimistic anyway, because it was picked on the folds that scored it.

3. **OOB is a free second opinion on CV.** When OOB and CV agree within the fold spread — as they do here on both cases — you have two estimates from two different schemes corroborating each other. Watch for divergence: it is the visible sign that the rows are not independent (time series, grouped data, or panel data).

4. **Four importance methods, four perspectives.** Linear-coef (what the linear baseline thinks), MDI (cheap, measured on the training rows), permutation (model-agnostic, measured here on held-out folds, with a fold-to-fold spread), drop-column (unique contribution, expensive — and itself a paired comparison of the forest with and without the column). Agreement is a strong signal; **disagreement is information about the data** — particularly the **substitution effect**, where a feature ranks high on MDI and permutation but low on drop-column because correlated alternatives step in when it is removed.

5. **Per-case champions — climbed as a ladder.**
   - **Regression (California Housing — HomeValue Analytics):** the **default random forest replaces OLS** — mean gain about +0.20 R², corrected interval [+0.18, +0.22], robust across the band. The chain is **OLS → forest**: nb11's depth-5 tree was unresolved against OLS and never displaced it.
   - **Classification (Wisconsin Breast Cancer — State Health Department):** **`LogReg(C=1.0)` stays** — the forest's comparison with it is unresolved (mean gain −0.006, interval [−0.019, +0.007]), and so is the tree's. The four-method importance table from §7 still gives the oncologists the interpretability story they asked for.

**Bridge to nb13 — Gradient Boosting:**

Random forests reduce variance; gradient boosting reduces **bias**. Where forests fit many trees in parallel and average them, boosting fits trees **sequentially** — each new tree is fit to the *residuals* of the running ensemble. On regression those residuals are the literal *(y − ŷ)* differences; on classification they are the gradient of the log-loss, which is essentially *(y − p̂)* — the gap between the true label and the running ensemble's predicted probability. In plain English: each new tree learns *"where are we still wrong on the data, and by how much?"*

The same dual-case pattern continues in nb13: `GradientBoostingClassifier` on Wisconsin breast cancer, `GradientBoostingRegressor` on California Housing, with the same folds, the same margin band and the same ladder. The current champions it has to beat are the ones you just named — `LogReg(C=1.0)` for MedScreen and the default forest for HomeValue. Bring today's tuning discipline with you: boosting adds `learning_rate` as a joint partner of `n_estimators` and `max_depth`, its grid also produces a nominee that must beat the default GBM, and the wrong combination overfits quickly. Boosting also amplifies leaky features more aggressively than any algorithm in the course, so nb13 closes with a leakage warning.

> **A question that often comes up at this point:** *"Does gradient boosting always beat random forest?"* No — and nb13 does not assume it will. A boosted model is often strong on tabular data once its `learning_rate × n_estimators` budget is tuned, but whether it earns a *worthwhile* gain over the forest on these folds is exactly the comparison nb13 runs, with the same `compare_paired` rule and the same declared margin. The trade-offs are real too: boosting is sequential (slower to fit, harder to parallelize), more sensitive to hyperparameters, and more prone to overfitting. nb13 walks both the potential gains and the trade-offs.

---


## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete both PAUSE-AND-DO exercises** — Exercise 1 (RF tuning, classification) and Exercise 2 (RF tuning, regression).
2. **Run All Cells** — `Runtime → Run all` to ensure every cell executes without error.
3. **Save a Copy** — `File → Save a copy in Drive`, or download as `.ipynb`.
4. **Submit** — upload the `.ipynb` file to the Notebook 12 participation assignment on Brightspace.

### Before Submitting, Check:

- [ ] Exercise 1 produces a tuning heatmap with the nominee starred, and both exercises print a `compare_paired` verdict against the default forest
- [ ] The four-method importance heatmap renders for both cases
- [ ] All figures render (none broken)
- [ ] Both `_clf` and `_reg` variable namespaces stay disjoint (no `NameError`)

### Next Step:

- **Notebook 13** — Gradient Boosting

---

---

## Bibliography

- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2023). *An Introduction to Statistical Learning with Python* — Ch. 8 (Tree-Based Methods), §8.2 Bagging, Random Forests, Boosting. Springer.
- Breiman, L. (2001). "Random Forests." *Machine Learning*, 45(1), 5–32.
- Breiman, L. (1996). "Bagging predictors." *Machine Learning*, 24(2), 123–140.
- Strobl, C., Boulesteix, A.-L., Zeileis, A., & Hothorn, T. (2007). "Bias in random forest variable importance measures: Illustrations, sources and a solution." *BMC Bioinformatics*, 8(25).
- scikit-learn User Guide — [Ensemble methods: Random Forests](https://scikit-learn.org/stable/modules/ensemble.html#random-forests)
- scikit-learn User Guide — [Permutation feature importance](https://scikit-learn.org/stable/modules/permutation_importance.html)
- Nadeau, C., & Bengio, Y. (2003). "Inference for the generalization error." *Machine Learning*, 52, 239–281. (The corrected interval in `compare_paired`.)
- Cawley, G. C., & Talbot, N. L. C. (2010). "On over-fitting in model selection and subsequent selection bias in performance evaluation." *Journal of Machine Learning Research*, 11, 2079–2107. (Why a grid's best cell is optimistic.)

---

<center>

**Thank you!**

</center>